What is a generator?

A generator is a function that uses yield instead of return. It produces values one at a time and pauses between each one — it remembers its state between calls.

In [1]:
#Generator function vs regular function:

In [2]:
# Regular function — computes ALL values, stores in memory:
def get_squares(n):
    result = []
    for i in range(n):
        result.append(i**2)
    return result   # returns entire list at once

# Generator function — produces ONE value at a time:
def gen_squares(n):
    for i in range(n):
        yield i**2   # pause here, remember state, resume next call

In [3]:
#How yield works:

In [4]:
def count_up(start, stop):
    print("Generator started")
    current = start
    while current <= stop:
        print(f"About to yield {current}")
        yield current              # pause here, return value
        print(f"Resumed after {current}")  # runs on next()
        current += 1
    print("Generator finished")

gen = count_up(1, 3)
print(next(gen))   # Generator started → About to yield 1 → 1
print(next(gen))   # Resumed after 1 → About to yield 2 → 2
print(next(gen))   # Resumed after 2 → About to yield 3 → 3
# next(gen)        # Resumed after 3 → Generator finished → StopIteration

Generator started
About to yield 1
1
Resumed after 1
About to yield 2
2
Resumed after 2
About to yield 3
3


In [5]:
#Generator expressions — one-liners:

In [6]:
# List comprehension (stores all in memory):
squares_list = [x**2 for x in range(1000000)]   # 8MB!

# Generator expression (stores ONE value):
squares_gen = (x**2 for x in range(1000000))    # ~200 bytes!

# Use exactly the same way:
print(next(squares_gen))   # 0
print(sum(squares_gen))    # works!

0
333332833333500000


In [7]:
#Memory comparison:

In [8]:
import sys
lst = [x**2 for x in range(10000)]
gen = (x**2 for x in range(10000))
print(sys.getsizeof(lst))   # ~87632 bytes
print(sys.getsizeof(gen))   # 208 bytes

85176
208


In [9]:
#yield from — delegate to sub-generator:

In [10]:
def gen_a():
    yield 1
    yield 2

def gen_b():
    yield 3
    yield 4

def combined():
    yield from gen_a()   # delegate to gen_a
    yield from gen_b()   # then gen_b
    yield from range(5,8)  # works with any iterable!

print(list(combined()))   # [1,2,3,4,5,6,7]

[1, 2, 3, 4, 5, 6, 7]


In [11]:
#send() — two-way communication:

In [12]:
def accumulator():
    total = 0
    while True:
        value = yield total   # yield current total, receive new value
        total += value

gen = accumulator()
next(gen)          # prime the generator (advance to first yield)
print(gen.send(10))   # 10
print(gen.send(20))   # 30
print(gen.send(5))    # 35

10
30
35


In [13]:
#throw() and close():

In [14]:
def safe_gen():
    try:
        yield 1
        yield 2
        yield 3
    except ValueError as e:
        print(f"Caught: {e}")
        yield -1

gen = safe_gen()
print(next(gen))              # 1
print(gen.throw(ValueError, "oops"))  # Caught: oops → -1
gen.close()                   # closes generator

1
Caught: oops
-1


In [15]:
#Infinite generators:

In [16]:
def natural_numbers():
    n = 1
    while True:
        yield n
        n += 1

def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

from itertools import islice
print(list(islice(fibonacci(), 10)))   # [0,1,1,2,3,5,8,13,21,34]

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


In [17]:
#examples

In [18]:
#First generators

In [23]:
# 1. countdown(n)
def countdown(n):
    while n > 0:
        yield n
        n -= 1
    yield "Blast off!"


# 2. even_numbers(start, stop)
def even_numbers(start, stop):
    for num in range(start, stop + 1):
        if num % 2 == 0:
            yield num


# 3. squares_gen(n)
def squares_gen(n):
    for i in range(1, n + 1):
        yield i * i


# 4. repeat_gen(value, times)
def repeat_gen(value, times):
    for _ in range(times):
        yield value


# 5. alternate(seq1, seq2)
def alternate(seq1, seq2):
    for a, b in zip(seq1, seq2):
        yield a
        yield b


# ----------------------------
# Testing with list()
# ----------------------------
print(list(countdown(5)))
print(list(even_numbers(1, 10)))
print(list(squares_gen(5)))
print(list(repeat_gen("hi", 3)))
print(list(alternate([1, 2, 3], ["a", "b", "c"])))


# ----------------------------
# Testing with next()
# ----------------------------
sq = squares_gen(5)

print("First square:", next(sq))
print("Second square:", next(sq))

[5, 4, 3, 2, 1, 'Blast off!']
[2, 4, 6, 8, 10]
[1, 4, 9, 16, 25]
['hi', 'hi', 'hi']
[1, 'a', 2, 'b', 3, 'c']
First square: 1
Second square: 4


In [19]:
#Generator expressions vs list comprehensions

In [24]:
import sys
from itertools import islice


# Helper function: Prime check
def is_prime(n):
    if n < 2:
        return False

    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False

    return True


# Helper function: Digit sum
def digit_sum(n):
    return sum(int(d) for d in str(n))


data = list(range(1, 1001))

# ==================================================
# 1. List comprehension vs Generator expression
# ==================================================

list_comp = [x * x for x in data]
gen_expr = (x * x for x in data)

print("List size:", sys.getsizeof(list_comp), "bytes")
print("Gen size:", sys.getsizeof(gen_expr), "bytes")


# ==================================================
# 2. Using generators with sum(), max(), min()
# ==================================================

sum_div7 = sum(x * x for x in data if x % 7 == 0)

max_prime_square = max(x * x for x in data if is_prime(x))

count_digit_sum = sum(
    1 for x in data
    if digit_sum(x) > 10
)

print("\nSum of squares (div by 7):", sum_div7)
print("Max prime square ≤ 1000:", max_prime_square)
print("Numbers with digit sum > 10:", count_digit_sum)


# ==================================================
# 3. Chaining generator expressions
# ==================================================

nums = range(1, 101)

primes = (x for x in nums if is_prime(x))
prime_squares = (x * x for x in primes)
filtered = (x for x in prime_squares if x > 200)

first_five = list(islice(filtered, 5))

print("\nFirst 5 prime squares > 200:", first_five)


# ==================================================
# 4. Generator can be iterated only once
# ==================================================

g = (x * x for x in range(1, 6))

print("\nFirst iteration:", list(g))
print("Generator exhausted:", list(g))

List size: 8856 bytes
Gen size: 208 bytes

Sum of squares (div by 7): 47262215
Max prime square ≤ 1000: 994009
Numbers with digit sum > 10: 717

First 5 prime squares > 200: [289, 361, 529, 841, 961]

First iteration: [1, 4, 9, 16, 25]
Generator exhausted: []


In [20]:
#yield from and infinite generators

In [25]:
from itertools import islice
from heapq import merge


# ==================================================
# 1. Recursive flatten using yield from
# ==================================================

def flatten_gen(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten_gen(item)
        else:
            yield item


# ==================================================
# 2. Infinite Generators
# ==================================================

# Prime number generator (infinite)
def primes():
    n = 2

    while True:
        is_prime = True

        for i in range(2, int(n ** 0.5) + 1):
            if n % i == 0:
                is_prime = False
                break

        if is_prime:
            yield n

        n += 1


# Powers of 2 generator (infinite)
def powers_of_2():
    value = 1

    while True:
        yield value
        value *= 2


# Collatz sequence generator
def collatz(n):
    while n != 1:
        yield n

        if n % 2 == 0:
            n //= 2
        else:
            n = 3 * n + 1

    yield 1


# ==================================================
# 3. Merge sorted iterables
# ==================================================

def merge_sorted(*iterables):
    yield from merge(*iterables)


# ==================================================
# 4. Custom chain using yield from
# ==================================================

def chain_gen(*iterables):
    for iterable in iterables:
        yield from iterable


# ==================================================
# Testing
# ==================================================

# Flatten
nested = [[1, [2, 3]], [4, [5, [6]]]]
print("Flattened:", list(flatten_gen(nested)))

# First 10 primes
print("First 10 primes:",
      list(islice(primes(), 10)))

# First 8 powers of 2
print("First 8 powers of 2:",
      list(islice(powers_of_2(), 8)))

# Collatz sequence
c = list(collatz(27))
print("Collatz(27) length:", len(c))

# Merge sorted streams
merged = merge_sorted(
    [1, 4, 7],
    [1, 2, 5, 8],
    [3, 6, 9]
)

print("Merged sorted:", list(merged))

# Custom chain
chained = chain_gen(
    [1, 2],
    [3, 4],
    [5, 6]
)

print("Chained:", list(chained))

Flattened: [1, 2, 3, 4, 5, 6]
First 10 primes: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
First 8 powers of 2: [1, 2, 4, 8, 16, 32, 64, 128]
Collatz(27) length: 112
Merged sorted: [1, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Chained: [1, 2, 3, 4, 5, 6]


In [21]:
#Generator pipelines for data processing

In [26]:
import random
import sys
import heapq


# ==================================================
# Data Source (simulates huge file)
# ==================================================

def read_records():
    random.seed(42)

    names = ["Amit", "Diya", "Raj", "Sara", "Neha",
             "Karan", "Riya", "Vikram", "Anjali", "Dev"]

    depts = ["Eng", "HR", "Sales", "Finance"]

    for i in range(1, 21):
        yield {
            "id": i,
            "name": f"{random.choice(names)}_{i}",
            "score": random.randint(50, 100),
            "dept": random.choice(depts)
        }


# ==================================================
# Stage 1: Parse / Clean
# ==================================================

def parse(records):
    for rec in records:

        if not isinstance(rec["score"], int):
            continue

        rec["name"] = rec["name"].strip()

        yield rec


# ==================================================
# Stage 2: Filter Department
# ==================================================

def filter_dept(records, dept):
    for rec in records:
        if rec["dept"] == dept:
            yield rec


# ==================================================
# Stage 3: Enrich
# ==================================================

def enrich(records):
    for rec in records:

        score = rec["score"]

        if score >= 90:
            grade = "A+"
        elif score >= 80:
            grade = "A"
        elif score >= 70:
            grade = "B"
        elif score >= 60:
            grade = "C"
        else:
            grade = "D"

        rec["grade"] = grade

        yield rec


# ==================================================
# Stage 4: Running Aggregation
# ==================================================

def aggregate(records):
    count = 0
    total = 0

    for rec in records:
        count += 1
        total += rec["score"]

        yield {
            "record": rec,
            "count": count,
            "avg_score": round(total / count, 2)
        }


# ==================================================
# Stage 5: Top N
# ==================================================

def top_n(records, n):
    heap = []

    count_seen = 0

    for rec in records:
        count_seen += 1

        if len(heap) < n:
            heapq.heappush(heap, (rec["score"], rec))
        else:
            heapq.heappushpop(heap, (rec["score"], rec))

    top_records = sorted(heap, reverse=True)

    for _, rec in top_records:
        yield rec

    print(f"\nTotal Eng records seen: {count_seen}")


# ==================================================
# Build Pipeline
# ==================================================

source = read_records()
parsed = parse(source)
eng_only = filter_dept(parsed, "Eng")
enriched = enrich(eng_only)

print("Memory usage (generator objects):")
print("read_records :", sys.getsizeof(source), "bytes")
print("parse        :", sys.getsizeof(parsed), "bytes")
print("filter_dept  :", sys.getsizeof(eng_only), "bytes")
print("enrich       :", sys.getsizeof(enriched), "bytes")

results = top_n(enriched, 3)

print("\nPipeline results (Eng dept, top 3):")

for rank, rec in enumerate(results, start=1):
    print(
        f"#{rank} "
        f"{rec['name']} | "
        f"Score:{rec['score']} | "
        f"Grade:{rec['grade']}"
    )

print("\nAll processed lazily — no full list in memory!")

Memory usage (generator objects):
read_records : 248 bytes
parse        : 224 bytes
filter_dept  : 216 bytes
enrich       : 232 bytes

Pipeline results (Eng dept, top 3):
#1 Diya_3 | Score:93 | Grade:A+
#2 Vikram_15 | Score:84 | Grade:A
#3 Sara_6 | Score:82 | Grade:A

Total Eng records seen: 5

All processed lazily — no full list in memory!


In [22]:
#send(), throw() and generator coroutines

In [28]:
from datetime import datetime


# ==================================================
# Custom Exception
# ==================================================

class ShutdownSignal(Exception):
    pass


# ==================================================
# Log Levels
# ==================================================

LEVELS = {
    "DEBUG": 10,
    "INFO": 20,
    "WARNING": 30,
    "ERROR": 40
}


# ==================================================
# Logger Coroutine
# ==================================================

def logger_gen(min_level="WARNING"):
    threshold = LEVELS[min_level]

    try:
        while True:
            level, message = yield

            if LEVELS[level] >= threshold:
                year = datetime.now().year
                print(f"[{level}] {year}: {message}")

    except ShutdownSignal:
        print("Shutdown signal received")


# ==================================================
# Test
# ==================================================

logger = logger_gen("WARNING")
next(logger)  # prime coroutine

logger.send(("DEBUG", "Variable x=10"))
logger.send(("INFO", "Application started"))
logger.send(("WARNING", "Disk space low"))
logger.send(("ERROR", "System crash!"))

# Graceful shutdown
try:
    logger.throw(ShutdownSignal)
except StopIteration:
    pass

[WARNING] 2026: Disk space low
[ERROR] 2026: System crash!
Shutdown signal received
